# Deploy Credit Score Model ke SageMaker Endpoint

Jalankan notebook ini **setelah** `pipeline_aws.py` selesai dan `best_model.pkl` sudah ada di `credit_scoring/model/`.

Urutan: Package model → Upload S3 → Uji lokal → Deploy endpoint → Smoke test → **Hapus endpoint**

Endpoint `ml.m5.large` menimbulkan biaya selama menyala. Setelah screenshot selesai diambil, langsung jalankan bagian terakhir (Cleanup).

In [ ]:
# Versi scikit-learn harus sama dengan saat model dilatih (framework 1.4-2 di pipeline_aws.py)
!pip install scikit-learn==1.4.2 --only-binary=:all: --quiet

In [ ]:
import json
import os
import tarfile

import boto3
import sagemaker

# ---- Konfigurasi: semua cell di bawah memakai nilai ini ----
REGION            = "us-east-1"
ENDPOINT_NAME     = "credit-score-endpoint"
INSTANCE_TYPE     = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"
MODEL_PKL_PATH    = "/home/ec2-user/SageMaker/credit_scoring/model/best_model.pkl"
MODEL_S3_PREFIX   = "credit-score"
# -------------------------------------------------------------

boto3.setup_default_session(region_name=REGION)
sm_session = sagemaker.Session()
BUCKET     = sm_session.default_bucket()
ROLE_ARN   = boto3.client("iam").get_role(RoleName="LabRole")["Role"]["Arn"]

print("Bucket  :", BUCKET)
print("Endpoint:", ENDPOINT_NAME)
print("Model   :", MODEL_PKL_PATH, "| ada:", os.path.exists(MODEL_PKL_PATH))

## Step 1 — Package model jadi .tar.gz
SageMaker Endpoint menerima model dalam bentuk arsip `.tar.gz`, bukan `.pkl` polos.

In [ ]:
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(MODEL_PKL_PATH, arcname="best_model.pkl")

with open(os.path.join(os.path.dirname(MODEL_PKL_PATH), "tuning_summary.json")) as f:
    ringkasan = json.load(f)
print("✅ model.tar.gz berhasil dibuat")
print(f"Model terbaik: {ringkasan['winner']} | CV F1 macro: {ringkasan['winner_cv_f1_macro']}")

## Step 2 — Upload model ke S3
Endpoint mengambil model dari S3, bukan dari folder lokal.

In [ ]:
MODEL_S3_URI = sm_session.upload_data("model.tar.gz", bucket=BUCKET, key_prefix=MODEL_S3_PREFIX)
print("✅ Model diupload ke", MODEL_S3_URI)

## Step 3 — Uji handler endpoint secara lokal
Sebelum membuat endpoint (yang lama dan berbayar), fungsi di `inference_aws.py` diuji langsung di notebook dengan tiga nasabah contoh. Ketiganya berasal dari data uji dan sama dengan contoh di `inferencing.py` versi lokal.

Kolom kategori dikirim sebagai **teks asli** ("Accountant", "Good", dst). Encoding dilakukan oleh model sendiri.

In [ ]:
from inference_aws import FEATURE_NAMES, input_fn, model_fn, output_fn, predict_fn

CONTOH_NASABAH = {
    "Good": {
        "Age": 47, "Annual_Income": 78039.48, "Monthly_Inhand_Salary": 6236.29,
        "Num_Bank_Accounts": 5, "Num_Credit_Card": 7, "Interest_Rate": 4,
        "Num_of_Loan": 4, "Delay_from_due_date": 21, "Num_of_Delayed_Payment": 17,
        "Changed_Credit_Limit": 2.59, "Num_Credit_Inquiries": 1,
        "Outstanding_Debt": 1188.91, "Credit_Utilization_Ratio": 28.62,
        "Credit_History_Age_Months": 344, "Total_EMI_per_month": 236.24,
        "Amount_invested_monthly": 97.54, "Monthly_Balance": 529.85,
        "Occupation": "Accountant", "Credit_Mix": "Good",
        "Payment_of_Min_Amount": "No", "Payment_Behaviour": "High_spent_Large_value_payments",
    },
    "Standard": {
        "Age": 48, "Annual_Income": 27109.79, "Monthly_Inhand_Salary": 2224.15,
        "Num_Bank_Accounts": 3, "Num_Credit_Card": 5, "Interest_Rate": 15,
        "Num_of_Loan": 3, "Delay_from_due_date": 26, "Num_of_Delayed_Payment": 12,
        "Changed_Credit_Limit": 9.26, "Num_Credit_Inquiries": 1,
        "Outstanding_Debt": 320.07, "Credit_Utilization_Ratio": 28.58,
        "Credit_History_Age_Months": 184, "Total_EMI_per_month": 43.38,
        "Amount_invested_monthly": 71.97, "Monthly_Balance": 387.06,
        "Occupation": "Entrepreneur", "Credit_Mix": "Standard",
        "Payment_of_Min_Amount": "No", "Payment_Behaviour": "Low_spent_Medium_value_payments",
    },
    "Poor": {
        "Age": 32, "Annual_Income": 53751.21, "Monthly_Inhand_Salary": 4260.27,
        "Num_Bank_Accounts": 9, "Num_Credit_Card": 8, "Interest_Rate": 19,
        "Num_of_Loan": 9, "Delay_from_due_date": 16, "Num_of_Delayed_Payment": 21,
        "Changed_Credit_Limit": 8.98, "Num_Credit_Inquiries": 12,
        "Outstanding_Debt": 3351.48, "Credit_Utilization_Ratio": 28.91,
        "Credit_History_Age_Months": 94, "Total_EMI_per_month": 222.26,
        "Amount_invested_monthly": 145.59, "Monthly_Balance": 318.17,
        "Occupation": "Developer", "Credit_Mix": "Bad",
        "Payment_of_Min_Amount": "Yes", "Payment_Behaviour": "High_spent_Small_value_payments",
    },
}

# Format request endpoint: {"instances": [[21 nilai sesuai urutan FEATURE_NAMES], ...]}
payload = json.dumps({"instances": [[kasus[k] for k in FEATURE_NAMES] for kasus in CONTOH_NASABAH.values()]})

with tarfile.open("model.tar.gz", "r:gz") as tar:
    tar.extractall("model_package")
model = model_fn("model_package")
hasil_lokal = json.loads(output_fn(predict_fn(input_fn(payload, "application/json"), model), "application/json")[0])

for label_asli, label_tebakan in zip(CONTOH_NASABAH, hasil_lokal["labels"]):
    print(f"Label asli: {label_asli:8s} | tebakan model: {label_tebakan}")

## Step 4 — Deploy ke SageMaker Endpoint
Proses ini memakan waktu sekitar 5–8 menit. Hanya `inference_aws.py` dan `preprocessing_aws.py` yang dikirim ke container, karena endpoint membersihkan input dengan fungsi `clean()` yang sama seperti saat training.

In [ ]:
from sagemaker.sklearn.model import SKLearnModel

# Endpoint configuration sisa run sebelumnya (dengan nama yang sama) membuat deploy gagal
# dengan error "Cannot create already existing endpoint configuration". Kalau tidak ada
# endpoint aktif yang memakainya, config lama itu dihapus dulu.
sm = boto3.client("sagemaker", region_name=REGION)
aktif = [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]]
config_lama = [c["EndpointConfigName"] for c in sm.list_endpoint_configs(NameContains=ENDPOINT_NAME)["EndpointConfigs"]]
if ENDPOINT_NAME in config_lama and ENDPOINT_NAME not in aktif:
    sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Config lama dihapus:", ENDPOINT_NAME)

model_endpoint = SKLearnModel(
    model_data=MODEL_S3_URI,
    role=ROLE_ARN,
    entry_point="inference_aws.py",
    dependencies=["preprocessing_aws.py"],
    framework_version=FRAMEWORK_VERSION,
    sagemaker_session=sm_session,
)
model_endpoint.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
status = boto3.client("sagemaker").describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
print(f"\n✅ Endpoint {ENDPOINT_NAME}: {status}")

## Step 5 — Smoke test
Tiga nasabah yang sama dikirim ke endpoint. Hasilnya harus sama dengan uji lokal di Step 3.

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=REGION)
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=payload,
)
hasil_endpoint = json.loads(response["Body"].read().decode("utf-8"))

for i, label_asli in enumerate(CONTOH_NASABAH):
    peluang = dict(zip(["Good", "Standard", "Poor"], hasil_endpoint["probabilities"][i]))
    print(f"Label asli: {label_asli:8s} | endpoint: {hasil_endpoint['labels'][i]:8s} | "
          + " ".join(f"{k} {v:.0%}" for k, v in peluang.items()))

assert hasil_endpoint["labels"] == hasil_lokal["labels"], "Hasil endpoint berbeda dengan uji lokal!"
print("\n✅ Hasil endpoint sama dengan uji lokal")

Kalau ada error, log container ada di CloudWatch Logs: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logsV2:log-groups (grup `/aws/sagemaker/Endpoints/credit-score-endpoint`).

## Cleanup — Hapus endpoint (WAJIB setelah screenshot selesai)
Endpoint, endpoint configuration, dan model dihapus berdasarkan nama, jadi cell ini tetap berhasil walaupun kernel sempat di-restart.

In [ ]:
sm = boto3.client("sagemaker", region_name=REGION)

try:
    config_name = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointConfigName"]
    model_names = [v["ModelName"] for v in sm.describe_endpoint_config(EndpointConfigName=config_name)["ProductionVariants"]]
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
    sm.delete_endpoint_config(EndpointConfigName=config_name)
    for nama in model_names:
        sm.delete_model(ModelName=nama)
    print(f"✅ Dihapus: endpoint {ENDPOINT_NAME}, config {config_name}, model {model_names}")
except sm.exceptions.ClientError as e:
    print("Tidak ada yang dihapus:", e.response["Error"]["Message"])

sisa = [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]]
print("Endpoint yang masih ada:", sisa or "tidak ada")